# Application Flow and Architectural Overview

This notebook provides a detailed walkthrough of how data and logic flow through the **Practise Application**. It is intended for developers to understand the sequence of operations from the entry point to the database.

## 1. Application Entry Point

The application starts at `PractiseApplication.java`.

**Key Responsibility:**
- Detect the current working directory.
- Load environment variables from `.env` via `Dotenv`.
- Set those values as system properties so Spring Boot can resolve placeholders in `application.yml`.
- Bootstrap the Spring Context.

## 2. Environment Loading and Startup Behavior

The application first checks the current working folder for a `.env` file.
If it is not found there, it falls back to `practise/.env`.

This fallback is important because the app may be launched from either:
- `/home/rishabh-developer/Documents/java backend/practise`
- `/home/rishabh-developer/Documents/java backend`

The loaded environment variables include:
- `DATABASE_URL`
- `DATABASE_USERNAME`
- `DATABASE_PASSWORD`
- `JWT_SECRET`

These variables are then bound into `application.yml` using Spring placeholders.

## 3. Layered Architecture Flow

The application follows a strict unidirectional flow for every request:

1.  **Client Request**: A JSON payload is sent to a REST endpoint.
2.  **Controller**: Captures the request, performs validation (`@Valid`), and maps it to a **DTO** or **Entity**.
3.  **Service**: Contains the business logic. It coordinates between repositories, applies validation, and handles transactions.
4.  **Repository**: Communicates with the PostgreSQL database using JPA/Hibernate.
5.  **Database**: Persists or retrieves the data.

## 4. Authentication and Authorization Flow

### Login / Signup
- `/api/auth/signup` creates a new user with a BCrypt-hashed password.
- `/api/auth/login` authenticates a user and returns a JWT token.

### JWT Filter
- `JwtAuthFilter` reads the `Authorization` header from incoming requests.
- If the header starts with `Bearer`, it extracts the token and validates it.
- The filter loads the username from the token, retrieves the user from the database, and sets the security context.
- If JWT validation fails, the filter returns `401 Unauthorized`.

### Security Configuration
- `WebSecurityConfig` allows unauthenticated access to `/api/auth/**` and Swagger endpoints.
- All other endpoints require authentication.
- Stateless session management is enabled with `SessionCreationPolicy.STATELESS`.

## 5. Detailed Example: Placing an Order

The order placement is the most complex flow in the system. Here is the step-by-step logic inside `OrderService.createOrder`:

### Step A: The Request (DTO)
The user sends an `OrderRequest` which contains customer details and a list of `OrderItemRequest` objects (Product ID + Quantity).

### Step B: Logic Execution
1.  **Initialization**: Create a new `Order` entity and set basic info.
2.  **Validation Loop**: For each item in the request:
    - Fetch the `Product` from `ProductRepository`.
    - **Check Stock**: If `stockQuantity < requestedQuantity`, throw a `RuntimeException`.
    - **Price Calculation**: Calculate the sub-total using `BigDecimal` for precision.
    - **Inventory Update**: Subtract the requested quantity from the product's stock.
3.  **Entity Mapping**: Create `OrderItem` entities linked to both the `Order` and the `Product`.
4.  **Finalization**: Set the total price and the list of items on the `Order` object.

### Step C: Transactional Save
The method is marked `@Transactional`. If any step fails (e.g., the last item in a 10-item list is out of stock), the database rolls back everything. No partial orders are created, and stock is not deducted incorrectly.

## 6. API Endpoints Reference

### Product API (`/api/products`)

| Method | Endpoint | Description | Payload |
| :--- | :--- | :--- | :--- |
| POST | `/` | Create a new product | `Product` (Entity) |
| GET | `/` | List all products | N/A |
| GET | `/{id}` | Get specific product | N/A |
| PUT | `/{id}` | Update product details | `Product` (Entity) |
| DELETE | `/{id}` | Remove a product | N/A |

### Order API (`/api/orders`)

| Method | Endpoint | Description | Payload |
| :--- | :--- | :--- | :--- |
| POST | `/` | Place a new order | `OrderRequest` (DTO) |
| GET | `/` | Get all orders | N/A |
| GET | `/{id}` | Get order by ID | N/A |

### Authentication API (`/api/auth`)

| Method | Endpoint | Description | Payload |
| :--- | :--- | :--- | :--- |
| GET | `/login` | Authenticate and receive JWT | `LoginRequest` |
| POST | `/signup` | Create a new user | `LoginRequest` |

## 7. Summary of Key Services

### `ProductService`
- **Purpose**: Direct CRUD management.
- **Key Logic**: Handles product retrieval, updates, creation, and deletion.

### `OrderService`
- **Purpose**: Complex business orchestration.
- **Key Logic**:
    - Precision math with `BigDecimal`.
    - Cross-entity coordination (linking `Order` with `Product` via `OrderItem`).
    - Stock deduction and transactional integrity.

### `AuthService`
- **Purpose**: User authentication and registration.
- **Key Logic**:
    - Authenticate credentials using `AuthenticationManager`.
    - Generate JWT tokens.
    - Hash passwords before saving new users.

## 8. Error Handling Strategy

The application implements a robust error handling strategy to ensure API reliability:

1.  **Request Validation**:
    - Usage of `@Valid` in Controllers triggers automatic 400 Bad Request responses if DTO constraints are violated.
2.  **Authentication and JWT Errors**:
    - `JwtAuthFilter` returns `401 Unauthorized` for invalid or missing tokens.
    - `GlobalExceptionHandler` handles authentication exceptions, JWT exceptions, access denied errors, and generic server errors.
3.  **Resource Discovery**:
    - Service layer queries use `.orElseThrow()` to surface missing resource errors.
4.  **Business Logic Guardrails**:
    - Explicit checks prevent invalid stock updates and incorrect order creation.
5.  **Transaction Integrity**:
    - `@Transactional` ensures rollback on failure.

## 9. Data Integrity Notes

- **DTO vs Entity**: `OrderRequest` is used for incoming data to prevent users from manually setting fields like `id` or `totalPrice`.
- **Circular References**: `@JsonManagedReference` and `@JsonBackReference` are used in entities to prevent infinite loops when Jackson serializes the Order <-> OrderItem relationship to JSON.

## 10. Running the App

From the `practise/` folder:

```bash
cd "/practise"
./mvnw spring-boot:run
```

The startup logic also supports launching from the parent directory by locating `.env` in the `practise` subfolder.
